# MC-5 lab: keyword spotting features and training in Python

**Goal.** You compute the Mel-frequency cepstral coefficients of a clip
of one second with numpy only, and you train a small classifier on them
without any tool of Edge Impulse Studio.

**Deliverable.** The table of section 9. Copy it into `report.md`.

**Time.** 50 minutes, sections 0 to 9.

**How to work.** Run the cells in order. A cell with the mark
`TODO (student)` has a task. The notebook has four tasks, and the cell
after a task checks your result. A check prints `complete` or
`not complete`.


In [ ]:
import os
import platform
import sys

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # no start message of TensorFlow

import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

tf.get_logger().setLevel("ERROR")

print("python    ", sys.version.split()[0])
print("numpy     ", np.__version__)
print("tensorflow", tf.__version__)
print("keras     ", keras.__version__)
print("machine   ", platform.machine())

RATE = 16000         # samples per second
CLIPS = 16000        # samples in one clip: 1 s
FRAME = 320          # samples in one frame: 20 ms
STRIDE = 320         # samples between two frames: 20 ms
NFFT = 512           # points of the FFT: 257 bins of 31.25 Hz
N_BINS = NFFT // 2 + 1
N_MEL = 32           # triangular filters of the Mel scale
N_MFCC = 13          # cepstral coefficients kept for each frame
ALPHA = 0.97         # factor of the pre-emphasis filter
CLASSES = ["no", "noise", "unknown", "yes"]
N_FRAMES = 1 + (CLIPS - FRAME) // STRIDE

# The lab folder. The notebook of the folder solutions/ uses the folder
# above, because the data and the code are one level up.
LAB = "." if os.path.isdir("host") and os.path.isdir("solutions") else ".."
KEYWORDS = os.path.join(LAB, "keywords", "clips")


def clip_files():
    '''Return the names of the clip files, sorted.'''
    if not os.path.isdir(KEYWORDS):
        return []
    return sorted(name for name in os.listdir(KEYWORDS)
                  if name.endswith(".wav"))


if not clip_files():
    print("No clip in the folder keywords/clips/. Run the script first,")
    print("from the folder of this lab:")
    print("    python3 host/get_keywords.py")
    raise SystemExit(1)
print()
print("clip folder:", os.path.normpath(KEYWORDS), "with",
      len(clip_files()), "clips")
print("frames per clip:", N_FRAMES, "values per clip:",
      N_FRAMES * N_MFCC)


## 1. Load the clips (4 min)

Each clip is one second of audio at 16 kHz. The script of the folder
`host/` wrote the amplitudes of each clip into one text file, one value
per line, so the notebook needs no audio package.

The four classes are `no`, `noise`, `unknown`, and `yes`. A clip of the
class `unknown` is a word that is not `yes` or `no`.

You do not change this code.


In [ ]:
def read_clip(path):
    '''Return the amplitudes of one clip as float32 in the range -1 to 1.'''
    with open(path, encoding="ascii") as handle:
        values = np.array(handle.read().split(), dtype=np.float32)
    return values / 32768.0


def load_all():
    '''Return the clips and the class of each clip.'''
    clips, labels = [], []
    for name in clip_files():
        clips.append(read_clip(os.path.join(KEYWORDS, name)))
        labels.append(CLASSES.index(name.split(".")[0]))
    return np.array(clips, dtype=np.float32), np.array(labels, dtype="int32")


X_audio, y = load_all()
print("clips:", X_audio.shape, " one clip has", X_audio.shape[1],
      "values, that is", X_audio.shape[1] // RATE, "s")
for k, label in enumerate(CLASSES):
    print("  %-8s %4d clips" % (label, int((y == k).sum())))
print()
print("One clip gives", X_audio.shape[1], "values. A model that reads")
print("them needs that many inputs.")


## 2. The steps that the notebook gives you (3 min)

The pipeline of a keyword spotting system has six steps: the
pre-emphasis, the framing, the windowing, the FFT, the Mel filters with
the logarithm, and the cosine transform. The cells below give the steps
that need no decision from you. Read them: two of the four tasks use
them.


In [ ]:
def frames(clip, frame=FRAME, stride=STRIDE):
    '''Cut the clip into frames. The last values are dropped when they do
    not fill a complete frame.'''
    count = 1 + (len(clip) - frame) // stride
    return np.array([clip[i * stride:i * stride + frame]
                     for i in range(count)])


def hamming(n):
    '''Return a Hamming window of n values. It is 1 in the middle and
    close to 0 at the two ends.'''
    return 0.54 - 0.46 * np.cos(2.0 * np.pi * np.arange(n) / (n - 1))


def power_spectrum(frame, window=None, nfft=NFFT):
    '''Return the power of each bin of one windowed frame. The window
    makes the ends of the frame small, so they do not spread their
    energy over every bin.'''
    if window is None:
        window = hamming(len(frame))
    spectrum = np.fft.rfft(frame * window, nfft)
    return np.abs(spectrum) ** 2 / nfft


def hz_to_mel(hz):
    '''Return the position of a frequency on the Mel scale.'''
    return 2595.0 * np.log10(1.0 + np.asarray(hz, dtype=float) / 700.0)


def mel_to_hz(mel):
    '''Return the frequency of a position on the Mel scale.'''
    return 700.0 * (10.0 ** (np.asarray(mel, dtype=float) / 2595.0) - 1.0)


def dct(values, n_out=N_MFCC):
    '''Return the first n_out coefficients of the discrete cosine
    transform of each row of values.'''
    n_in = values.shape[1]
    k = np.arange(n_out)
    m = np.arange(n_in)
    basis = np.cos(np.pi * k[:, None] * (2 * m[None, :] + 1) / (2 * n_in))
    return (values @ basis.T) / n_in


def log_mel(clip, filters, nfft=NFFT):
    '''Return the energy of each Mel filter of each frame of one clip:
    one row per frame and one column per filter.'''
    emphasised = np.append(clip[0], clip[1:] - ALPHA * clip[:-1])
    window = hamming(FRAME)
    power = np.array([power_spectrum(block, window, nfft)
                      for block in frames(emphasised)])
    return np.log(power @ filters.T + 1e-10)


## 3. The Mel filter bank (12 min)

**Task 1.** Write `mel_filters(n_filters, n_bins, f_min, f_max,
sample_rate)`. It returns a matrix with one row per filter and one
column per bin of the FFT.

Each filter is a triangle on the Mel scale: it rises from 0 to 1 over
its left half, and falls from 1 to 0 over its right half. Two filters
touch at the half point. The first filter starts at `f_min` and the
last one ends at `f_max`.

The Mel scale gives the low frequencies more filters than the high
ones, because the ear resolves the low frequencies better.

The two functions that convert the scale are given above.


In [ ]:
# TODO (student) Task 1: the triangular filters of the Mel scale.
def mel_filters(n_filters, n_bins, f_min, f_max, sample_rate):
    '''Return the matrix of the filters: one row per filter, one column
    per bin of the FFT. Every filter rises from 0 to 1 over its left
    half and falls from 1 to 0 over its right half.'''
    return np.zeros((n_filters, n_bins), dtype=np.float32)


In [ ]:
def check_task_1():
    '''Print the result of task 1. The check calculates the expected
    properties itself: one maximum of 1 for each filter, no negative
    value, a peak that moves to the right from one filter to the next,
    and a last filter near the top of the spectrum.'''
    filters = mel_filters(N_MEL, N_BINS, 0.0, RATE / 2, RATE)
    ok = filters.shape == (N_MEL, N_BINS) and bool(np.all(filters >= 0))
    peaks, heights = [], []
    if ok:
        peaks = [int(np.argmax(row)) for row in filters]
        heights = [float(np.max(row)) for row in filters]
        ok = (ok and all(abs(h - 1.0) < 1e-6 for h in heights)
              and all(b > a for a, b in zip(peaks, peaks[1:]))
              and peaks[-1] > 0.8 * N_BINS)
    print("Task 1:", "complete" if ok else "not complete")
    print("  the matrix has", filters.shape[0], "filters of",
          filters.shape[1], "bins")
    print("  the last filter must reach bin", int(0.8 * N_BINS),
          "or above")
    if ok:
        print("  the peak of the first filter is at bin", peaks[0],
              "and of the last one at bin", peaks[-1])
        print("  the width of the first filter is",
              int(np.count_nonzero(filters[0])),
              "bins and of the last one",
              int(np.count_nonzero(filters[-1])))


check_task_1()


## 4. The cepstral coefficients (12 min)

**Task 2.** Write `mfcc(clip, filters=None)`. It returns the features of
one clip: one row per frame and `N_MFCC` columns.

The steps are: the pre-emphasis, the framing, the Hamming window, the
FFT, the power of each bin, the energy of each Mel filter, the
logarithm, and the cosine transform. The functions of section 2 do all
of them except the filters and the last two steps.

The logarithm is inside `log_mel`, so you do not write it.


In [ ]:
# TODO (student) Task 2: the cepstral coefficients of one clip.
def mfcc(clip, filters=None):
    '''Return the features of one clip: one row per frame and N_MFCC
    columns.'''
    return np.zeros((N_FRAMES, N_MFCC), dtype=np.float32)


In [ ]:
def check_task_2():
    '''Print the result of task 2. The check uses the number of frames,
    the number of columns, the fact that the values are finite, and the
    fact that the first coefficient carries the energy of the clip, so
    its mean is large in absolute value. It does not call the function
    of the task.'''
    clip = X_audio[0]
    got = np.asarray(mfcc(clip), dtype=float)
    ok = (got.shape == (N_FRAMES, N_MFCC)
          and bool(np.all(np.isfinite(got)))
          and abs(float(np.mean(got[:, 0]))) > abs(float(np.mean(got[:, -1]))))
    print("Task 2:", "complete" if ok else "not complete")
    print("  the matrix has", got.shape[0], "rows and", got.shape[1],
          "columns")
    print("  the answer needs", N_FRAMES, "rows and", N_MFCC, "columns")
    if got.shape == (N_FRAMES, N_MFCC):
        print("  the mean of the first coefficient is %.3f and of the "
              "last one %.3f" % (np.mean(got[:, 0]),
                                 np.mean(got[:, -1])))


check_task_2()


## 5. Look at one clip (3 min)

The three plots show what each step keeps. The waveform shows the
samples. The log Mel energies show the filters, which are narrow at the
low frequencies. The cepstral coefficients show the envelope of the
spectrum, which is what the model reads.


In [ ]:
filters = mel_filters(N_MEL, N_BINS, 0.0, RATE / 2, RATE)
index = int(np.argmax(y == CLASSES.index("yes")))
clip = X_audio[index]
energies = log_mel(clip, filters)
coefficients = dct(energies, N_MFCC)

figure, axes = plt.subplots(3, 1, figsize=(6.0, 5.0), sharex=True)
t = np.arange(CLIPS) / RATE
axes[0].plot(t, clip, linewidth=0.4, color="#4C72B0")
axes[0].set_ylabel("amplitude")
axes[0].set_title("clip of the class yes: waveform, log Mel energies, "
                  "MFCC", fontsize=8)
image = axes[1].imshow(energies.T, aspect="auto", origin="lower",
                       cmap="magma")
axes[1].set_ylabel("Mel filter")
axes[1].set_xlabel("frame (20 ms)")
figure.colorbar(image, ax=axes[1], fraction=0.03)
axes[2].imshow(coefficients.T, aspect="auto", origin="lower",
               cmap="RdBu_r")
axes[2].set_ylabel("coefficient")
figure.tight_layout()
plt.show()
print("one clip:", N_FRAMES, "frames,", N_MEL, "log Mel values,",
      N_FRAMES * N_MFCC, "cepstral values")


## 6. The features of every clip (6 min)

The cell computes the features of all the clips and writes them into
the file `features.npz`. The next run reads that file, so the second run
of the notebook is fast.

Then it splits the clips into a training set and a test set. The split
is random, so a clip of a speaker can be in both sets. The accuracy of
the test set is then better than the accuracy on new speakers.

You do not change this code.


In [ ]:
import time

CACHE = os.path.join(LAB, "features.npz")
start = time.time()
if os.path.exists(CACHE):
    data = np.load(CACHE)
    X_mfcc, X_mel, y = data["mfcc"], data["mel"], data["y"]
    print("read the features from", os.path.normpath(CACHE))
else:
    features, mel_values = [], []
    for number, clip in enumerate(X_audio):
        features.append(mfcc(clip, filters))
        mel_values.append(log_mel(clip, filters))
        if (number + 1) % 100 == 0:
            print("  %d of %d clips" % (number + 1, len(X_audio)))
    X_mfcc = np.array(features, dtype=np.float32)
    X_mel = np.array(mel_values, dtype=np.float32)
    np.savez(CACHE, mfcc=X_mfcc, mel=X_mel, y=y)
    print("wrote", os.path.normpath(CACHE))
print("the features of the clips:", X_mfcc.shape, "in %.1f s"
      % (time.time() - start))
print("the log Mel energies of the clips:", X_mel.shape)
print("the values of one clip:", X_mfcc.shape[1] * X_mfcc.shape[2],
      "against", CLIPS, "for the raw audio")


def split(x, ratio=0.8):
    '''Return two index arrays: the training set and the test set. The
    split is the same for every run, because the seed is fixed.'''
    order = np.random.default_rng(0).permutation(len(y))
    cut = int(ratio * len(y))
    return order[:cut], order[cut:]


train_index, test_index = split(X_mfcc)
print()
print("training clips:", len(train_index), " test clips:", len(test_index))


## 7. The classifier (10 min)

**Task 3.** Write `make_model(n_frames, n_features)`. The model reads
the 50 frames of one clip with 13 values each.

Three layers:

1. a `Conv1D` layer with 16 filters of 3 frames and ReLU, then a max
   pooling over the time with a size of 2;
2. a `Conv1D` layer with 32 filters of 3 frames and ReLU, then the same
   max pooling;
3. a `Dense` layer with 4 outputs and softmax.

A convolution with three frames and no padding removes two frames, so
the two pooling layers leave 11 frames for the last convolution.

Give the layers the names `conv_1`, `conv_2`, and `dense`, and the input
the name `mfcc_input`. Without the names, the size of the file of the
model changes with each model that the notebook builds.


In [ ]:
# TODO (student) Task 3: the classifier.
def make_model(n_frames, n_features):
    '''Return the model of the task description.'''
    return keras.Sequential(name="keyword")


In [ ]:
def check_task_3():
    '''Print the result of task 3. The check calculates the expected
    number of parameters itself, with the length that each convolution
    and each pooling layer leaves. A convolution with three frames and
    no padding removes two frames.'''
    first, second = 16, 32
    n_features = X_mfcc.shape[2]
    after_first = N_FRAMES - 2
    after_pool_1 = (after_first - 2) // 2 + 1
    after_second = after_pool_1 - 2
    after_pool_2 = (after_second - 2) // 2 + 1
    expected = ((3 * n_features * first + first)
                + (3 * first * second + second)
                + after_pool_2 * second * len(CLASSES) + len(CLASSES))
    try:
        model = make_model(N_FRAMES, n_features)
        params = model.count_params()
        shape = model.output_shape[-1]
    except Exception as error:                # noqa: BLE001
        print("Task 3: not complete")
        print("  the model raises an error:", error)
        return False
    ok = params == expected and shape == len(CLASSES)
    print("Task 3:", "complete" if ok else "not complete")
    print("  the input has %d frames of %d values, and %d frames after "
          "the two convolutions and the two pooling layers"
          % (N_FRAMES, n_features, after_pool_2))
    print("  parameters %d (expected %d), outputs %d"
          % (params, expected, shape))
    return ok


MODEL_COMPLETE = check_task_3()


## 8. Train (10 min)

The cells below train the model on the cepstral coefficients. The mean
and the standard deviation of each feature come from the training clips
only.

You do not change this code.


In [ ]:
def normalize(x_train, x_test):
    '''Return the two arrays, each value scaled with the mean and the
    standard deviation of the training values.'''
    mean = x_train.mean(axis=(0, 1), keepdims=True)
    std = x_train.std(axis=(0, 1), keepdims=True)
    std[std == 0.0] = 1.0
    return (x_train - mean) / std, (x_test - mean) / std


def fit_and_score(x_train, y_train, x_test, y_test, epochs=60):
    '''Train the model and return the test accuracy and the number of
    parameters. The cell does nothing until the check of task 3 passes.'''
    if not MODEL_COMPLETE:
        return float("nan"), 0
    keras.utils.set_random_seed(0)
    model = make_model(x_train.shape[1], x_train.shape[2])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    model.fit(x_train, y_train, epochs=epochs, batch_size=32, verbose=0,
              shuffle=True)
    scores = model.evaluate(x_test, y_test, verbose=0)
    return float(scores[1]), model.count_params()


def percent(value):
    '''Return an accuracy for the report, or the text that says that the
    number was not measured.'''
    return ("not measured" if value != value
            else "%.1f percent" % (100 * value))


mfcc_train, mfcc_test = normalize(X_mfcc[train_index],
                                  X_mfcc[test_index])
accuracy_mfcc, parameters_mfcc = fit_and_score(
    mfcc_train, y[train_index], mfcc_test, y[test_index])
print("the cepstral coefficients: %d values per clip, %d parameters, "
      "test accuracy %s" % (mfcc_train.shape[1] * mfcc_train.shape[2],
                            parameters_mfcc, percent(accuracy_mfcc)))


## 9. The cepstrum against the spectrogram (10 min)

The log Mel energies are a spectrogram with a coarser frequency axis.
The cosine transform mixes them into 13 coefficients per frame, so the
values of different filters become comparable.

**Task 4.** Before you run the next cell, complete the table with the
number of values that one clip gives for each input, and write the input
that you expect to give the better test accuracy.


In [ ]:
# TODO (student) Task 4: the number of values of one clip.
values_per_clip = {"raw audio": 0, "log Mel spectrogram": 0,
                   "cepstral coefficients": 0}
best_input = "write one of the three names here"


In [ ]:
mel_train, mel_test = normalize(X_mel[train_index], X_mel[test_index])
accuracy_mel, parameters_mel = fit_and_score(
    mel_train, y[train_index], mel_test, y[test_index])

audio_train = X_audio[train_index].reshape(len(train_index), CLIPS, 1)
audio_test = X_audio[test_index].reshape(len(test_index), CLIPS, 1)
accuracy_audio, parameters_audio = fit_and_score(
    audio_train, y[train_index], audio_test, y[test_index])

print("raw audio               : %5d values, %6d parameters, %s"
      % (CLIPS, parameters_audio, percent(accuracy_audio)))
print("log Mel spectrogram     : %5d values, %6d parameters, %s"
      % (N_FRAMES * N_MEL, parameters_mel, percent(accuracy_mel)))
print("cepstral coefficients   : %5d values, %6d parameters, %s"
      % (N_FRAMES * N_MFCC, parameters_mfcc, percent(accuracy_mfcc)))


In [ ]:
def check_task_4():
    '''Print the result of task 4. The check calculates the three
    numbers itself.'''
    expected = {"raw audio": CLIPS,
                "log Mel spectrogram": N_FRAMES * N_MEL,
                "cepstral coefficients": N_FRAMES * N_MFCC}
    ok = (values_per_clip == expected
          and best_input in expected)
    print("Task 4:", "complete" if ok else "not complete")
    print("  values per clip:", values_per_clip)
    print("  the input you expect:", best_input)
    if ok and accuracy_mfcc == accuracy_mfcc:
        print("  in this run the cepstral coefficients reach",
              percent(accuracy_mfcc), "and the spectrogram",
              percent(accuracy_mel))


check_task_4()


## 10. The table for the report (3 min)

Copy this table into `report.md`, then answer the three questions of the
report. Use the numbers of your own run.


In [ ]:
table = [("raw audio", CLIPS, parameters_audio, accuracy_audio),
         ("log Mel spectrogram", N_FRAMES * N_MEL, parameters_mel,
          accuracy_mel),
         ("cepstral coefficients", N_FRAMES * N_MFCC, parameters_mfcc,
          accuracy_mfcc)]

print("| Input | Values per clip | Parameters | Test accuracy |")
print("|---|---|---|---|")
for name, values, parameters, accuracy in table:
    print("| %s | %d | %d | %s |" % (name, values, parameters,
                                     percent(accuracy)))
print()
print("%d training clips and %d test clips. The split is random, so a"
      % (len(train_index), len(test_index)))
print("clip of a speaker can be in both sets. The accuracies come from a")
print("laptop CPU, so they say nothing about the time on a board.")

# The file that the figure script of the course repository reads.
import json
results = {"rate": RATE, "frames": N_FRAMES, "n_mel": N_MEL,
           "n_mfcc": N_MFCC, "clips": len(y),
           "inputs": [{"name": name, "values": values,
                       "parameters": parameters, "accuracy": accuracy}
                      for name, values, parameters, accuracy in table]}
with open(os.path.join(LAB, "results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=1)
print()
print("The notebook wrote results.json for the figures of the deck.")


## Questions for `report.md`

1. Which input gives the smallest model, and how many parameters does it
   save compared with the raw audio?
2. The first cepstral coefficient is the energy of the whole clip and
   the last one is small. What do the 13 coefficients describe, and why
   does the cosine transform make them comparable?
3. The split of the clips is random, and a clip of one speaker can be
   in the training set and in the test set. What does that do to the
   test accuracy, and what would you change for a product?
